## Dataset Extraction

In [1]:
import zipfile

zip_ref = zipfile.ZipFile('archive (1).zip', 'r')
zip_ref.extractall('data')
zip_ref.close()

## Dataset Verification

In [2]:
import os
print(os.listdir('data'))

['metadata', 'test.txt', 'train.txt', 'valid.txt']


### Data Loading

In [3]:
file = open('data/train.txt', 'r')
data = file.readlines()

print(data[:20])

['-DOCSTART- -X- -X- O\n', '\n', 'EU NNP B-NP B-ORG\n', 'rejects VBZ B-VP O\n', 'German JJ B-NP B-MISC\n', 'call NN I-NP O\n', 'to TO B-VP O\n', 'boycott VB I-VP O\n', 'British JJ B-NP B-MISC\n', 'lamb NN I-NP O\n', '. . O O\n', '\n', 'Peter NNP B-NP B-PER\n', 'Blackburn NNP I-NP I-PER\n', '\n', 'BRUSSELS NNP B-NP B-LOC\n', '1996-08-22 CD I-NP O\n', '\n', 'The DT B-NP O\n', 'European NNP I-NP B-ORG\n']


In [4]:
import os
print(os.getcwd())

C:\Users\Pooja\OneDrive\Desktop\Innomatics_internship\GenAI Tasks


### Data Preprocessing

In [5]:
sentences = []
labels = []

sentence = []
label = []

for line in data:
    if line.strip() == "":
        if sentence:
            sentences.append(sentence)
            labels.append(label)
            sentence = []
            label = []
    else:
        parts = line.split()
        if len(parts) >= 2:
            sentence.append(parts[0])   # word
            label.append(parts[1])      # POS tag

In [6]:
print(sentences[1])
print(labels[1])

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.']


### Label Mapping

In [7]:
unique_labels = list(set(label for sublist in labels for label in sublist))

label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

### Tokenization

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

### Tokenization & Label Alignment

In [9]:
def tokenize_and_align_labels(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        is_split_into_words=True,
        max_length=128   
    )

    aligned_labels = []

    for i in range(len(sentences)):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[labels[i][word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [10]:
encoded_data = tokenize_and_align_labels(sentences, labels)

### Dataset Creation

In [11]:
import torch

class POSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.encodings["labels"][idx])
        }

    def __len__(self):
        return len(self.encodings["input_ids"])

In [12]:
small_data = {key: val[:500] for key, val in encoded_data.items()}  
train_dataset = POSDataset(small_data)

#### Model Selection

In [13]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

#### Training Configuration

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,   
    num_train_epochs=1,               
    logging_steps=10,
    save_strategy="no"
)

#### Model Training

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [16]:
trainer.train()

C:\Users\Pooja\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.587466
20,3.205529
30,3.076461


TrainOutput(global_step=32, training_loss=3.268332377076149, metrics={'train_runtime': 655.3055, 'train_samples_per_second': 0.763, 'train_steps_per_second': 0.049, 'total_flos': 32675087616000.0, 'train_loss': 3.268332377076149, 'epoch': 1.0})

In [17]:
sentence = ["John works at Google"]

inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs).logits
predictions = outputs.argmax(dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, pred in zip(tokens, predictions[0]):
    print(token, id2label[pred.item()])

[CLS] NNP
john NNP
works NNP
at NNP
google NNP
[SEP] .


## Conclusion

In this project, we built a token classification system using a BERT model.

We successfully:
- Preprocessed the dataset  
- Performed tokenization and label alignment  
- Trained a transformer model  
- Generated predictions  

This project highlights how transformer models can be applied to NLP tasks like POS tagging and chunking.